[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week08_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 8 Assignment: Sequential Testing (Wald's SPRT)

## QuickCart — Making Faster Experiment Decisions

QuickCart's experiments currently run for a fixed duration (usually 2-4 weeks). But sometimes the evidence is overwhelming after just a few days — a change either clearly helps or clearly hurts. The product team wants to know: **can we stop experiments early when the evidence is clear, without inflating false positive rates?**

The answer is yes — using **Sequential Testing**. Specifically, you'll implement **Wald's Sequential Probability Ratio Test (SPRT)**, which:
- Monitors evidence as data accumulates
- Stops as soon as there's enough evidence to decide (reject H0 or accept H0)
- Controls both Type I and Type II error rates at pre-specified levels

**But first, you need to understand WHY you can't just peek at a normal test repeatedly.**

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)

---

## Task 1: The Peeking Problem

### Context

A common mistake: run an A/B test, check daily if it's significant, and stop as soon as p < 0.05. This seems reasonable but dramatically inflates false positive rates.

**Why?** Each peek is an independent chance to cross the significance threshold by luck. Over 30 days of peeking, the probability of *ever* crossing 0.05 is much higher than 5%.

### Problem

Demonstrate the peeking problem:

1. Simulate 1000 AA tests (no true effect) where data arrives daily for 30 days
2. Each day, 100 new observations per group arrive
3. After each day's data arrives, run a t-test on ALL accumulated data
4. Track whether the test was EVER significant (p < 0.05) during the 30-day window

Report: What fraction of the 1000 null experiments would be incorrectly declared significant if we peek daily?

**Expected result:** ~20-25% false positive rate (vs. the nominal 5%)

<details>
<summary>Hint (click to expand)</summary>

```python
n_simulations = 1000
n_days = 30
observations_per_day = 100

ever_significant = 0

for sim in range(n_simulations):
    control_data = []
    treatment_data = []
    was_significant = False
    
    for day in range(n_days):
        # Add new day's data (both from same distribution = no effect)
        control_data.extend(np.random.normal(0, 1, observations_per_day))
        treatment_data.extend(np.random.normal(0, 1, observations_per_day))
        
        # Peek: run t-test on all accumulated data
        _, p_value = stats.ttest_ind(control_data, treatment_data)
        if p_value < 0.05:
            was_significant = True
            break  # stop peeking (we'd declare significance)
    
    if was_significant:
        ever_significant += 1

false_positive_rate = ever_significant / n_simulations
```

</details>

In [ ]:
# YOUR CODE HERE
#
# Simulate the peeking problem:
# 1. Run 1000 AA tests with daily peeking
# 2. Report the inflated false positive rate
# 3. (Optional) Also track how the cumulative FPR grows with number of peeks

n_simulations = 1000
n_days = 30
observations_per_day = 100
alpha = 0.05

# YOUR CODE HERE
pass

In [ ]:
# Verification (uncomment after completing):

# print(f"Nominal alpha: {alpha}")
# print(f"Actual false positive rate with daily peeking: {false_positive_rate:.1%}")
# print(f"Inflation factor: {false_positive_rate / alpha:.1f}x")

# assert false_positive_rate > 0.15, \
#     f"Expected FPR > 15% with peeking, got {false_positive_rate:.1%}. Check your simulation."
# assert false_positive_rate < 0.40, \
#     f"FPR seems too high ({false_positive_rate:.1%}). Check your simulation."
# print("\nTask 1 check passed! Peeking inflates false positives as expected.")

In [ ]:
# Optional: Plot how false positive rate grows with number of peeks
# YOUR CODE HERE
#
# Suggested: For each day d in [1, 30], compute the fraction of simulations
# that ever crossed p < 0.05 by day d. Plot this curve.
pass

---

## Task 2: Implement the Sequential Tester (SPRT)

### Context

Wald's SPRT solves the peeking problem by using **different boundaries** that account for multiple looks at the data. Instead of comparing a p-value to a fixed threshold, it tracks the cumulative **log-likelihood ratio**:

$$\Lambda_n = \sum_{i=1}^n \log \frac{f_1(x_i)}{f_0(x_i)}$$

where $f_0$ is the density under H0 and $f_1$ is the density under H1.

**Decision boundaries:**
- Upper boundary: $B = \log\frac{1-\beta}{\alpha}$ — if $\Lambda_n \geq B$, reject H0 (accept H1)
- Lower boundary: $A = \log\frac{\beta}{1-\alpha}$ — if $\Lambda_n \leq A$, reject H1 (accept H0)
- If $A < \Lambda_n < B$, continue collecting data

### Problem

Implement the `SequentialTester` class:

<details>
<summary>Hint 1: Computing log-likelihood ratio for Normal distributions</summary>

If H0: $X \sim N(\mu_0, \sigma^2)$ and H1: $X \sim N(\mu_1, \sigma^2)$, then:

$$\log \frac{f_1(x)}{f_0(x)} = \frac{(\mu_1 - \mu_0)}{\sigma^2} \left(x - \frac{\mu_0 + \mu_1}{2}\right)$$

For batch data (comparing means of two groups), you can compute the log-LR for the difference in means.

</details>

<details>
<summary>Hint 2: Using scipy pdf objects</summary>

The class takes `pdf_one` and `pdf_two` as scipy frozen distributions. You can compute log-likelihoods as:
```python
log_lr = np.sum(pdf_two.logpdf(data) - pdf_one.logpdf(data))
```
where `pdf_one` is the H0 distribution and `pdf_two` is the H1 distribution.

</details>

<details>
<summary>Hint 3: Handling the difference of means</summary>

For an A/B test, you're testing the *difference* between control and treatment means. A simple approach:
1. Compute the difference statistic for each batch: `mean(treatment) - mean(control)`
2. Under H0: this difference is ~N(0, sigma) 
3. Under H1: this difference is ~N(delta, sigma)
4. Accumulate the log-LR of these difference statistics

</details>

In [ ]:
class SequentialTester:
    """Wald's Sequential Probability Ratio Test (SPRT).
    
    Monitors cumulative log-likelihood ratio and makes a decision
    when it crosses upper or lower boundaries.
    """
    
    def __init__(self, metric_name: str, time_column_name: str, 
                 alpha: float, beta: float, pdf_one, pdf_two):
        """Initialize the sequential tester.
        
        Parameters
        ----------
        metric_name : str
            Name of the metric being tested.
        time_column_name : str
            Name of the time/batch column in data.
        alpha : float
            Type I error rate (probability of false positive).
        beta : float
            Type II error rate (probability of false negative).
        pdf_one : scipy.stats frozen distribution
            Distribution under H0 (null hypothesis).
        pdf_two : scipy.stats frozen distribution
            Distribution under H1 (alternative hypothesis).
        """
        # YOUR CODE HERE
        # Store parameters
        # Compute boundaries:
        #   upper_boundary = log((1 - beta) / alpha)
        #   lower_boundary = log(beta / (1 - alpha))
        # Initialize cumulative log-LR to 0
        pass
    
    def run_test(self, data_control: np.ndarray, data_pilot: np.ndarray) -> tuple:
        """Run a fresh test on complete data.
        
        Resets state and processes all data at once.
        
        Parameters
        ----------
        data_control : np.ndarray
            Control group observations.
        data_pilot : np.ndarray
            Treatment group observations.
        
        Returns
        -------
        tuple
            (result, length) where:
            - result: 1 = reject H0 (effect exists), 0 = reject H1 (no effect),
                      0.5 = insufficient data (boundaries not crossed)
            - length: number of observations processed before decision
        """
        # YOUR CODE HERE
        pass
    
    def add_data(self, data_control: np.ndarray, data_pilot: np.ndarray) -> tuple:
        """Add new batch of data and continue accumulating evidence.
        
        Does NOT reset state — continues from where the last call left off.
        If a decision has already been made, returns immediately without
        processing new data.
        
        Parameters
        ----------
        data_control : np.ndarray
            New control group observations.
        data_pilot : np.ndarray
            New treatment group observations.
        
        Returns
        -------
        tuple
            (result, length) — same format as run_test.
        """
        # YOUR CODE HERE
        pass

In [ ]:
# Test Task 2: Basic functionality

# Set up: testing for a shift of 0.5 in mean (Normal data with sigma=1)
# H0: difference = 0, H1: difference = 0.5
pdf_h0 = stats.norm(loc=0, scale=1)   # null: no difference
pdf_h1 = stats.norm(loc=0.5, scale=1) # alternative: difference of 0.5

tester = SequentialTester(
    metric_name='revenue_diff',
    time_column_name='day',
    alpha=0.05,
    beta=0.20,
    pdf_one=pdf_h0,
    pdf_two=pdf_h1
)

# Check boundaries
expected_upper = np.log((1 - 0.20) / 0.05)  # log(0.8/0.05) = log(16) ~ 2.77
expected_lower = np.log(0.20 / (1 - 0.05))  # log(0.2/0.95) ~ -1.56

print(f"Upper boundary: {tester.upper_boundary:.4f} (expected ~{expected_upper:.4f})")
print(f"Lower boundary: {tester.lower_boundary:.4f} (expected ~{expected_lower:.4f})")

assert abs(tester.upper_boundary - expected_upper) < 0.01, "Upper boundary incorrect"
assert abs(tester.lower_boundary - expected_lower) < 0.01, "Lower boundary incorrect"

# Test with clear signal (large effect)
np.random.seed(42)
control = np.random.normal(0, 1, 200)
treatment = np.random.normal(0.8, 1, 200)  # large effect, should detect quickly

result, length = tester.run_test(control, treatment)
print(f"\nClear signal test: result={result}, stopped at observation {length}")
assert result == 1, f"Should reject H0 with strong effect, got result={result}"
assert length < 200, f"Should stop early with strong signal, used {length} observations"

print("\nBasic Task 2 checks passed!")

In [ ]:
# Test batch processing (add_data)
tester2 = SequentialTester(
    metric_name='revenue_diff',
    time_column_name='day',
    alpha=0.05,
    beta=0.20,
    pdf_one=pdf_h0,
    pdf_two=pdf_h1
)

np.random.seed(42)
# Add data in batches (simulating daily arrivals)
final_result = 0.5
for day in range(30):
    batch_control = np.random.normal(0, 1, 50)
    batch_treatment = np.random.normal(0.5, 1, 50)  # real effect
    result, length = tester2.add_data(batch_control, batch_treatment)
    if result != 0.5:  # decision made
        final_result = result
        print(f"Decision made on day {day+1}: {'Reject H0' if result == 1 else 'Accept H0'}")
        print(f"Total observations processed: {length}")
        break

if final_result == 0.5:
    print("No decision after 30 days")

assert final_result == 1, "Should detect the real effect"
print("\nBatch processing check passed!")

---

## Task 3: Validate Error Rate Control

### Context

The SPRT promises to control Type I error at $\alpha$ and Type II error at $\beta$. Verify this empirically.

### Problem

Run two sets of simulations:

**Under H0 (no effect):** Run 1000 simulations where both groups are N(0,1). Check that the false positive rate is at most $\alpha$.

**Under H1 (real effect of 0.5):** Run 1000 simulations where treatment is N(0.5, 1). Check that the detection rate is at least $1 - \beta$.

Also report: What is the **average stopping time** compared to using all data? (This demonstrates the speed advantage of sequential testing.)

<details>
<summary>Hint</summary>

```python
# For each simulation:
# 1. Generate data in daily batches (e.g., 50 obs/group/day for up to 60 days)
# 2. Feed to SequentialTester using add_data
# 3. Record the result and stopping time

# Under H0:
n_simulations = 1000
max_days = 60
obs_per_day = 50

results_h0 = []
stopping_times_h0 = []

for sim in range(n_simulations):
    tester = SequentialTester(...)
    for day in range(max_days):
        control = np.random.normal(0, 1, obs_per_day)
        treatment = np.random.normal(0, 1, obs_per_day)  # H0: no effect
        result, length = tester.add_data(control, treatment)
        if result != 0.5:
            break
    results_h0.append(result)
    stopping_times_h0.append(length)

# Type I error = fraction where result == 1 (incorrectly rejected H0)
type_i_error = np.mean(np.array(results_h0) == 1)
```

</details>

In [ ]:
# YOUR CODE HERE
#
# 1. Under H0: Verify Type I error control
# 2. Under H1: Verify power (1 - Type II error)
# 3. Report average stopping time vs fixed sample size

n_simulations = 1000
max_days = 60
obs_per_day = 50
alpha = 0.05
beta = 0.20

# YOUR CODE HERE
pass

In [ ]:
# Verification (uncomment after completing):

# print("=== Under H0 (no effect) ===")
# print(f"Type I error rate: {type_i_error:.3f} (target: <= {alpha})")
# print(f"Average stopping time: {np.mean(stopping_times_h0):.0f} observations")

# print(f"\n=== Under H1 (effect = 0.5) ===")
# print(f"Power (detection rate): {power:.3f} (target: >= {1-beta})")
# print(f"Average stopping time: {np.mean(stopping_times_h1):.0f} observations")
# print(f"Max possible observations: {max_days * obs_per_day}")
# print(f"Speed-up factor: {max_days * obs_per_day / np.mean(stopping_times_h1):.1f}x")

# # Verify error control
# assert type_i_error <= alpha + 0.02, f"Type I error {type_i_error:.3f} exceeds alpha={alpha}"
# assert power >= (1 - beta) - 0.05, f"Power {power:.3f} below target {1-beta}"
# print("\nTask 3 checks passed! SPRT controls error rates correctly.")

---

## Task 4: Visualize the SPRT Path

### Context

A powerful way to communicate sequential testing to stakeholders is to show the **decision path**: how the log-likelihood ratio evolves over time, with the decision boundaries clearly marked.

### Problem

Create a visualization that shows:
1. The cumulative log-likelihood ratio over time (one observation at a time)
2. The upper boundary (horizontal line)
3. The lower boundary (horizontal line)
4. The point where the boundary is crossed (decision point)

Create two plots side by side:
- Left: A test under H1 (real effect) — should cross the upper boundary
- Right: A test under H0 (no effect) — should cross the lower boundary (or wander)

<details>
<summary>Hint</summary>

You'll need to track the cumulative log-LR at each observation. Modify your tester or compute it externally:

```python
# Generate data
np.random.seed(42)
n_obs = 200
control = np.random.normal(0, 1, n_obs)
treatment = np.random.normal(0.5, 1, n_obs)  # under H1

# Compute log-LR for each observation
# For the difference (treatment[i] - control[i]):
differences = treatment - control
log_lr_increments = pdf_h1.logpdf(differences) - pdf_h0.logpdf(differences)
cumulative_log_lr = np.cumsum(log_lr_increments)

# Plot
plt.plot(cumulative_log_lr)
plt.axhline(upper_boundary, color='green', linestyle='--', label='Reject H0')
plt.axhline(lower_boundary, color='red', linestyle='--', label='Accept H0')
```

</details>

In [ ]:
# YOUR CODE HERE
#
# Create side-by-side plots showing SPRT paths:
# Left: under H1 (real effect)
# Right: under H0 (no effect)
#
# Each plot should show:
# - Cumulative log-LR path (blue line)
# - Upper boundary (green dashed)
# - Lower boundary (red dashed)
# - Decision point marked (star or dot)
# - Shaded "continue" region between boundaries

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# ...
pass

**Your interpretation:**

*Describe what you observe in the two plots. How does the log-LR path behave differently under H0 vs H1? Why does SPRT stop early when there's a real effect?*

---

## Summary

In this assignment you:
- Demonstrated that naive peeking inflates false positive rates 4-5x
- Implemented Wald's SPRT with proper decision boundaries
- Verified that SPRT controls both Type I and Type II error rates
- Showed that SPRT reaches decisions faster than fixed-horizon tests
- Visualized the sequential decision process

**Key takeaways:**
1. Never peek at fixed-horizon tests repeatedly — use sequential methods if you want early stopping.
2. SPRT provides a principled framework: specify the errors you'll tolerate ($\alpha$, $\beta$) and the effect size you care about, and the test tells you when to stop.
3. The trade-off: SPRT requires specifying H1 in advance (what effect size to power for), and it can take longer than fixed-horizon when the true effect is much smaller than specified.